In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.visualization import ImageNormalize, AsinhStretch
import numpy as np
from vis import plot_model_residual, plot_data_psf
import numpy as np
from scipy.ndimage import map_coordinates, zoom

#### Here we see if the data image and the psf image have same dimensions or not

In [ ]:
## here my science cutout is at 0.05”/pix, while the PSF you want to use is on 0.063”/pixel.

data = fits.open("../data/RUBIES-BLAGN-1/F115W.fits")
psf = fits.open("../PSF_files/F115W.fits")

print(psf[1].header.get("PIXELSCL"))
print(abs(data[0].header.get("CD1_1"))*3600)             # pixel_scale = abs(-1.3888889e-05) * 3600 ≈ 0.05 arcsec/pixel

0.0311
0.049999999999996804


#### If dimensions are not same, we move forward to resample the psf file

In [3]:
# Science image pixel scale
pix_science = 0.05

# PSF detector-sampled pixel scale
pix_psf = 0.063

zoom_factor = pix_psf / pix_science   # 1.26

with fits.open(psf_file) as hdul:
    psf = hdul[1].data.astype(float)   # DET_SAMP extension
    hdr = hdul[1].header.copy()


psf_resampled = zoom(psf, zoom_factor, order=3)
psf_resampled /= np.sum(psf_resampled)          # Renormalize total flux

newhdr = fits.Header()
newhdr["PIXELSCL"] = pix_science
newhdr["COMMENT"] = "Resampled from DET_SAMP extension of F277W_psf.fits"

fits.PrimaryHDU(psf_resampled, header=newhdr).writeto("../PSF_files/F277W_psf_0p05.fits", overwrite=True) ### ADD THE PATH HERE

### TESTS
print("Original shape:", psf.shape)
print("Resampled shape:", psf_resampled.shape)
print("Sum:", psf_resampled.sum())

ypeak, xpeak = np.unravel_index(np.argmax(psf_resampled), psf_resampled.shape)
print("Peak pixel:", xpeak, ypeak)

NameError: name 'psf_file' is not defined

#### Now, we have all the data files to start with the fitting

In [ ]:
# First we need to check if all the POI needs to be cropped or not, if so, we crop it using 

data_new = data[y1:y2, x1:x2]
psf_new = psf[y1:y2, x1:x2]

### MAIN TASK: Update all the profiles -- gaussian, sersic or exponential ------

##### and you run the 'run_fit.sh' as,
##### chmod +x run_fit.sh
##### ./run_fit.sh J0100-2017 F115W point 0.13  (0.13 is the sky, so workaround with that)